# T17 박스 검출 YOLO nano 학습 (ADR-0004 2단계)

분할 검출기(`voss_vision.box_detect`, detector=seg)가 만든 **자동 라벨**로 YOLO nano 를 학습한다. 10/08 공용 PC 에서 같은 녹화로 분할과 비교한다.

1. 개인 PC 에서 만든 `t17_yolo_v1.zip`(train + holdout + holdout_check + data.yaml)을 팀 Drive `datasets/t17_yolo_v1/` 에 올린다(구조: `docs/setup/drive.md`). Colab 은 팀 공용 계정으로 연다.
2. 런타임 → 런타임 유형 변경 → **T4 GPU**. 위에서부터 차례로 실행. GPU 할당량을 계정 하나로 나눠 쓰므로 학습은 한 번에 한 명만, Slack 에 알린다.
3. 세션이 끊기면 Drive 마운트·데이터 준비 두 셀을 다시 돌린 뒤 **이어 학습** 셀만 실행한다(체크포인트는 Drive 에 있다).
4. 사람 확인을 마친 `check.csv` 는 Drive `datasets/t17_yolo_v1/check.csv` 에 올린다. 평가 셀은 그 파일을 먼저 쓰고, 없으면 zip 안의 빈 사본으로 잰다(참고값).

- holdout(C bag 전체 = 박스 통과 10회, B bag 900 프레임~)은 학습에 쓰지 않는다. 최종 비교는 사람 확인 세트(`holdout_check/`, 통과 10회 × 20장 + 음성 25장, `check.csv` 의 ok 칸) 기준 — ADR-0004 결정 2·3.
- 가중치(`.pt`, `.onnx`)는 레포에 커밋하지 않는다(규칙 8). Drive `models/` 에만 둔다(이름에 버전 `_v1`).
- 지연(≤ 20 ms)은 Colab 이 아니라 공용 PC 에서 잰다.

In [1]:
from google.colab import drive

drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/rokey_cobot2_VOSS"  # 팀 Drive 루트 (docs/setup/drive.md)
ZIP = f"{ROOT}/datasets/t17_yolo_v1/t17_yolo_v1.zip"
RUNS = f"{ROOT}/colab/t17/runs"  # 체크포인트 (세션 끊김 대비)
MODELS = f"{ROOT}/models"
RUN_NAME = "t17_y11n"
MODEL = "yolo11n.pt"  # YOLO nano (ADR-0004)
OUT_NAME = "box_yolo11n_v1"  # models/ 에 둘 이름 (버전 접미사)

Mounted at /content/drive


In [2]:
import torch
import yaml

assert torch.cuda.is_available(), "GPU 런타임이 아니다 — 런타임 → 런타임 유형 변경 → T4 GPU"
print(torch.cuda.get_device_name(0))
!pip -q install ultralytics
!mkdir -p /content/data && cd /content/data && unzip -q -o "$ZIP" && ls t17_yolo_v1
D = "/content/data/t17_yolo_v1"
# data.yaml 의 path: . 는 ultralytics 가 작업 폴더(/content) 기준으로 풀어 train/ 을 못 찾는다 → 절대 경로 사본
cfg = yaml.safe_load(open(f"{D}/data.yaml"))
cfg["path"] = D
DATA = f"{D}/data_colab.yaml"
yaml.safe_dump(cfg, open(DATA, "w"), allow_unicode=True)
print(open(DATA).read())

Tesla T4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.1/96.1 kB 11.1 MB/s eta 0:00:00
data.yaml  holdout  holdout_check  train
names:
  0: box
path: /content/data/t17_yolo_v1
train: train/images
val: holdout/images



## 학습 (체크포인트 5 epoch 마다 Drive 저장)

In [3]:
from ultralytics import YOLO

model = YOLO(MODEL)
model.train(
    data=DATA,
    imgsz=640,
    epochs=40,
    batch=16,
    seed=0,
    project=RUNS,
    name=RUN_NAME,
    exist_ok=True,
    save_period=5,
    fliplr=0.5,
    flipud=0.5,
    degrees=45,
    hsv_v=0.3,
)  # 박스 회전·밝기 변화 대비

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/t17_yolo_v1/data_colab.yaml, degrees=45, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, frac

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b54c3fdf150>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

## 이어 학습 (세션이 끊겼을 때만)

In [4]:
#from ultralytics import YOLO

#YOLO(f"{RUNS}/{RUN_NAME}/weights/last.pt").train(resume=True)

WARNING ⚠️ model '/content/drive/MyDrive/rokey_cobot2_VOSS/colab/t17/runs/t17_y11n/weights/last.pt' is not a resumable training checkpoint (missing epoch/optimizer state). Use 'resume' only to continue incomplete training. Starting new training instead.
Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco8.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0, 16, 17, 20, 25, 58])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b54397743d0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,

## holdout 평가: 박스 프레임 검출률(IoU ≥ 0.5), 빈 프레임 오검출

In [5]:
import csv
import glob
import os

from ultralytics import YOLO

best = YOLO(f"{RUNS}/{RUN_NAME}/weights/best.pt")


def load(p):
    t = p.replace("/images/", "/labels/")[:-4] + ".txt"
    return [list(map(float, line.split()[1:])) for line in open(t) if line.strip()]


def iou(a, b):  # cx, cy, w, h (정규화)
    ax0, ay0, ax1, ay1 = a[0] - a[2] / 2, a[1] - a[3] / 2, a[0] + a[2] / 2, a[1] + a[3] / 2
    bx0, by0, bx1, by1 = b[0] - b[2] / 2, b[1] - b[3] / 2, b[0] + b[2] / 2, b[1] + b[3] / 2
    iw, ih = max(0, min(ax1, bx1) - max(ax0, bx0)), max(0, min(ay1, by1) - max(ay0, by0))
    inter = iw * ih
    return inter / (a[2] * a[3] + b[2] * b[3] - inter + 1e-9)


def evaluate(img_dir, skip=()):
    hit = n_box = fp = n_neg = 0
    for p in sorted(glob.glob(f"{img_dir}/*.jpg")):
        if os.path.basename(p)[:-4] in skip:
            continue
        gt = load(p)
        r = best.predict(p, imgsz=640, conf=0.5, verbose=False)[0]
        pred = r.boxes.xywhn.cpu().numpy().tolist()
        if gt:
            n_box += len(gt)
            hit += sum(any(iou(g, q) >= 0.5 for q in pred) for g in gt)
        else:
            n_neg += 1
            fp += bool(pred)
    print(
        f"{img_dir}: 박스 검출률 {hit}/{n_box} = {100 * hit / max(n_box, 1):.2f}%  |  빈 프레임 오검출 {fp}/{n_neg}"
    )


evaluate(f"{D}/holdout/images")  # 자동 라벨 기준 (참고)
# 사람 확인 세트: check.csv 에서 ok 가 n 인 행(자동 라벨이 틀림)은 빼고 잰다 — 판정 기준
CHECK = f"{ROOT}/datasets/t17_yolo_v1/check.csv"  # 사람 확인본 (Drive)
if not os.path.exists(CHECK):
    CHECK = f"{D}/holdout_check/check.csv"  # zip 안 사본 — ok 칸이 비어 있다
rows = list(csv.DictReader(open(CHECK, encoding="utf-8-sig", errors="replace")))
ok = [r["ok(y/n)"].strip().lower() for r in rows]
print(f"{CHECK}: y {ok.count('y')} · n {ok.count('n')} · 빈칸 {ok.count('')} / {len(rows)}")
bad = {f"{r['bag']}_{int(r['frame']):05d}" for r, o in zip(rows, ok, strict=True) if o == "n"}
evaluate(f"{D}/holdout_check/images", skip=bad)
if ok.count(""):
    print("⚠️ 사람 확인 미완 — 위 결과는 참고값. 빈칸 0 인 check.csv 로 다시 잰다.")

print("주의: holdout 라벨은 자동 라벨이다. 최종 판정은 사람 확인 세트(check.csv) 결과로 한다.")

/content/data/t17_yolo_v1/holdout/images: 박스 검출률 835/835 = 100.00%  |  빈 프레임 오검출 0/129
/content/data/t17_yolo_v1/holdout_check/check.csv: y 0 · n 0 · 빈칸 225 / 225
/content/data/t17_yolo_v1/holdout_check/images: 박스 검출률 190/190 = 100.00%  |  빈 프레임 오검출 16/41
⚠️ 사람 확인 미완 — 위 결과는 참고값. 빈칸 0 인 check.csv 로 다시 잰다.
주의: holdout 라벨은 자동 라벨이다. 최종 판정은 사람 확인 세트(check.csv) 결과로 한다.


## ONNX 내보내기 → Drive models/

In [6]:
p = best.export(format="onnx", imgsz=640, opset=17, simplify=True)
os.makedirs(MODELS, exist_ok=True)
!cp "$p" "$MODELS/{OUT_NAME}.onnx" && ls -la "$MODELS"

Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/rokey_cobot2_VOSS/colab/t17/runs/t17_y11n/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (5.2 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 12 packages in 294ms
Prepared 4 packages in 551ms
Installed 4 packages in 44ms
 + colorama==0.4.6
 + onnx==1.23.2
 + onnxruntime==1.30.0
 + onnxslim==0.1.97

requirements: AutoUpdate success ✅ 1.2s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx